# Ad Ranking & CTR Prediction

Companion notebook for the [Ad Ranking & CTR Prediction lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/06-ad-ranking-and-ctr-prediction).

We implement eCPM-based ranking, a simplified FM (Factorization Machine) for CTR prediction, and second-price auction logic. Pure NumPy.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)

## 1 — eCPM ranking

Ads are ranked by expected CPM = predicted CTR × bid. This aligns ML quality with revenue.

In [ ]:
def rank_ads(bids, predicted_ctrs):
    """Rank ads by eCPM = CTR × bid × 1000."""
    ecpm = predicted_ctrs * bids * 1000
    order = np.argsort(-ecpm)
    return order, ecpm

# 5 advertisers
bids = np.array([5.00, 2.00, 8.00, 3.50, 1.50])
pred_ctrs = np.array([0.01, 0.08, 0.005, 0.04, 0.12])

order, ecpm = rank_ads(bids, pred_ctrs)
print("Advertiser | Bid  | Pred CTR | eCPM")
for i in order:
    print(f"  Ad {i}     | ${bids[i]:.2f} | {pred_ctrs[i]:.3f}    | ${ecpm[i]:.2f}")

## 2 — Second-price auction

The winner pays the second-highest bid (plus a small increment). This is the dominant mechanism in online advertising.

In [ ]:
def second_price_auction(bids, increment=0.01):
    """Returns (winner_id, payment)."""
    order = np.argsort(-bids)
    winner = order[0]
    second_bid = bids[order[1]]
    payment = second_bid + increment
    return winner, payment

# Auction on raw bids
winner, payment = second_price_auction(bids)
print(f"Winner: Ad {winner} (bid=${bids[winner]:.2f})")
print(f"Payment: ${payment:.2f} (second-highest bid + $0.01)")
print(f"Winner profit at true value $6: ${6 - payment:.2f}")

## 3 — Simplified Factorization Machine for CTR

A Factorization Machine models pairwise feature interactions via embedding dot products — essential for high-cardinality ad features.

In [ ]:
class FactorizationMachine:
    def __init__(self, n_features, d=4, seed=0):
        rng_ = np.random.default_rng(seed)
        self.w0 = 0.0                                  # global bias
        self.w  = np.zeros(n_features)                 # linear weights
        self.V  = rng_.normal(0, 0.01, (n_features, d))  # interaction factors

    def predict(self, x):
        """x: (n_features,) sparse binary feature vector"""
        linear = self.w0 + self.w @ x
        # Pairwise interaction term: 0.5 * (||Vx||^2 - ||V^2 x||_sum)
        Vx = self.V.T @ x                              # (d,)
        interaction = 0.5 * (Vx @ Vx - (self.V**2).T @ (x**2) @ np.ones(4))
        return 1 / (1 + np.exp(-(linear + interaction)))  # sigmoid -> CTR

# Features: [user_age_bucket, device_mobile, ad_category, time_evening, ...]
n_features = 20
fm = FactorizationMachine(n_features=n_features)

# Simulate a feature vector for a user-ad pair
x = np.zeros(n_features)
x[[0, 5, 12, 17]] = 1                                 # active features

ctr_pred = fm.predict(x)
print(f"Predicted CTR for this user-ad pair: {ctr_pred:.4f}")
print(f"eCPM at $3.50 bid: ${ctr_pred * 3.50 * 1000:.2f}")

## ✏️ Your turn

**Exercise.** Implement `gsp_auction(bid_quality_scores, slots)` for a Generalized Second Price auction with `slots` ad slots. Each ad in position $k$ pays the minimum bid that keeps it in position $k$ (i.e., just above the next-lower advertiser's bid-quality score).

Return an array of payments for the top `slots` winners.

In [ ]:
def gsp_auction(bid_quality_scores, slots=3):
    """
    bid_quality_scores: array of bid × quality_score for each advertiser
    slots: number of ad slots available
    Returns payments[k] for k in range(slots).
    """
    # TODO(you): sort by bid_quality_score descending, then compute each winner's payment
    # payment[k] = bid_quality_scores[k+1] / quality_score[k]  (simplified)
    # For simplicity, use: payment[k] = bid_quality_scores[k+1] + 0.01
    return ...

bqs = np.array([12.0, 9.0, 6.0, 3.0, 1.5])  # bid × quality scores
payments = gsp_auction(bqs, slots=3)
print("GSP payments for top-3 slots:", payments)

In [ ]:
# Assertion
pay = gsp_auction(bqs, slots=3)
assert len(pay) == 3, "Should return 3 payments for 3 slots"
assert pay[0] > pay[1] > pay[2] >= 0, "Higher slots should pay more"
assert abs(pay[0] - (bqs[1] + 0.01)) < 0.001, "Winner pays next competitor's score + increment"
print("✓ GSP auction correct")

<details>
<summary>Solution</summary>

```python
def gsp_auction(bid_quality_scores, slots=3):
    order = np.argsort(-bid_quality_scores)
    payments = []
    for k in range(slots):
        if k + 1 < len(bid_quality_scores):
            payments.append(bid_quality_scores[order[k + 1]] + 0.01)
        else:
            payments.append(0.01)
    return np.array(payments)
```

In GSP, each winner pays just enough to outbid the person below them — encouraging efficient allocation while being simpler than VCG.
</details>